### Initial inspection of Eng to cz dataset

In this inspection, we get these important information:

1. If the 20 documents in this dataset have desired mean length as our en-fr experiments. We filter out **8 documents** as the rest have either document lengths exceeding 2000 characters or below 1300 characters.

[3, 5, 6, 7, 13, 17, 18, 19] - document number / Translation_id

[1785, 1575, 1367, 1625, 1525, 1472, 1523, 1356] / doc lengths

2. We calculate the spans using XCOMET custom class and we have [73, 59, 41, 32, 90, 74, 42, 25] spans for each document. We can work with documents with spans 41, 32, 42, and 25 as they have reasonable many spans, but we have to be careful if the spans have at least 4 characters.

3. We do not create the dataset here for our purpose, we will build it in **statistics_translator_en_cz.ipynb** file.

**here its xcomet_frn_czech_both and in terminal its xcomet**

how to activate the env in the terminal:
conda activate envs/xcomet/
(xcomet) Singularity>

In [1]:
from datasets import load_dataset
data_wmt = load_dataset("zouharvi/optimal-reference-translations", 'ort_wmt')["train"]



In [2]:
data_wmt

Dataset({
    features: ['ref', 'systems', 'src'],
    num_rows: 160
})

In [3]:
print(data_wmt)


Dataset({
    features: ['ref', 'systems', 'src'],
    num_rows: 160
})


In [4]:
print(data_wmt.column_names)


['ref', 'systems', 'src']


In [5]:
print(data_wmt[0])

{'ref': {'R1': 'Vláda přikázala odchod do důchodu dalším 15 daňovým úředníkům v rámci čtvrtého balíčku opatření proti úředníkům obviněným z korupce a dalších nezákonných praktik.', 'R1_pe_layman_cardiff': 'Vláda přikázala odchod do důchodu dalším 15 daňovým úředníkům v rámci čtvrtého balíčku opatření proti úředníkům obviněným z korupce a dalších nezákonných praktik.', 'R1_pe_layman_funafuti': 'Vláda přikázala odchod do důchodu dalším 15 daňovým úředníkům v rámci čtvrtého balíčku opatření proti úředníkům obviněným z korupce a dalších nezákonných praktik.', 'R1_pe_layman_hanoi': 'Vláda přikázala odchod do důchodu dalším 15 daňovým úředníkům v rámci čtvrtého balíčku opatření proti úředníkům obviněným z korupce a dalších nezákonných praktik.', 'R1_pe_layman_washington': 'Vláda přikázala odchod do důchodu dalším 15 daňovým úředníkům v rámci čtvrtého balíčku opatření proti úředníkům obviněným z korupce a dalších nezákonných praktik.', 'R1_pe_professional_ankara': 'Vláda během čtvrté části zá

In [6]:
count = 1
for data in data_wmt:
    print("line no: ", count)
    print(data["src"])
    count += 1

line no:  1
The government has compulsorily retired 15 more tax officers in the fourth tranche of its crackdown on errant officials accused of corruption and other malpractices.
line no:  2
The Central Board of Indirect Taxes and Customs (CBIC) -- the agency that oversees GST and import tax collections -- compulsorily retired 15 senior officers under Fundamental Rule 56 (J) on corruption and other charges, official sources said.
line no:  3
Since June, this is the fourth round of sacking of corrupt tax officials. In the previous three rounds, 49 high ranking tax officers, including 12 from the Central Board of Direct Taxes (CBDT), were compulsorily retired.
line no:  4
Sources said the action was in line with Prime Minister Narendra Modi's address to the nation from the ramparts of the Red Fort when he had said some black sheep in the tax administration may have misused their powers and harassed taxpayers, either by targeting honest assesses or taking excessive action for minor or proc

In [7]:
df = data_wmt.to_pandas()

In [8]:
df.head()

,ref,systems,src
0,{'R1': 'Vláda přikázala odchod do důchodu dalš...,{'CUNI-DocTransformer.1450': {'score': 0.76862...,The government has compulsorily retired 15 mor...
1,{'R1': 'Ústřední rada nepřímých daní a cel (CB...,{'CUNI-DocTransformer.1450': {'score': 0.58643...,The Central Board of Indirect Taxes and Custom...
2,"{'R1': 'Od června je to už počtvrté, co jsou z...",{'CUNI-DocTransformer.1450': {'score': 0.63602...,"Since June, this is the fourth round of sackin..."
3,{'R1': 'Podle oficiálních zdrojů proběhla akce...,{'CUNI-DocTransformer.1450': {'score': -0.2728...,Sources said the action was in line with Prime...
4,"{'R1': '„Nedávno jsme podnikli odvážný krok, k...",{'CUNI-DocTransformer.1450': {'score': 0.37188...,"""We have recently taken the bold step of compu..."


The dataset only has three columns ref, systems, src. We need only R1 and online B and the the source as R1 is the postedited version of the online B sytem. We make a central dataset adding document number and line number for each document so that later we can trace back to it according to the line number we took for creating the updated dataset.

Analysis of the official WMT20 reference (REF1) revealed it was not translated "from scratch" as required by official guidelines. Using automated detection, researchers confirmed that **REF1 is a post-edited version of the ONLINE-B system.**

**R1 (P1/REF1) – Post-edited MT: Created by an agency using ONLINE-B as a base.** High quality, but structurally biased toward the MT system.

# add line number and document number to each line/ sentences

As each document contains 8 source sentences, and there are 20 documents, we add line numbers from 1 to 8 and document number from 1 to 20 inclusive.

In [9]:
df["document_id"] = df.index // 8 + 1
df["line_id"] = df.index % 8 + 1

In [10]:
print(df["document_id"].unique())
print(df["line_id"].unique())

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
[1 2 3 4 5 6 7 8]


In [11]:
df.head(24)

,ref,systems,src,document_id,line_id
0,{'R1': 'Vláda přikázala odchod do důchodu dalš...,{'CUNI-DocTransformer.1450': {'score': 0.76862...,The government has compulsorily retired 15 mor...,1,1
1,{'R1': 'Ústřední rada nepřímých daní a cel (CB...,{'CUNI-DocTransformer.1450': {'score': 0.58643...,The Central Board of Indirect Taxes and Custom...,1,2
2,"{'R1': 'Od června je to už počtvrté, co jsou z...",{'CUNI-DocTransformer.1450': {'score': 0.63602...,"Since June, this is the fourth round of sackin...",1,3
3,{'R1': 'Podle oficiálních zdrojů proběhla akce...,{'CUNI-DocTransformer.1450': {'score': -0.2728...,Sources said the action was in line with Prime...,1,4
4,"{'R1': '„Nedávno jsme podnikli odvážný krok, k...",{'CUNI-DocTransformer.1450': {'score': 0.37188...,"""We have recently taken the bold step of compu...",1,5
5,"{'R1': 'Téměř polovina úředníků, kteří odešli ...",{'CUNI-DocTransformer.1450': {'score': 0.28887...,Nearly half of the officials retired were thos...,1,6
6,{'R1': 'V červnu vláda donutila odejít do důch...,{'CUNI-DocTransformer.1450': {'score': 0.63602...,"In June, the government had compulsorily retir...",1,7
7,{'R1': 'V srpnu vláda poslala do důchodu 22 úř...,{'CUNI-DocTransformer.1450': {'score': 0.47106...,"In August, the government compulsorily retired...",1,8
8,{'R1': 'Žena v Maine obdržela během pěti dnů 5...,"{'CUNI-DocTransformer.1450': {'score': 0.0, 't...",A woman in Maine got 500 letters from United H...,2,1
9,"{'R1': 'Stephanie Layová uvedla, že během čtvr...","{'CUNI-DocTransformer.1450': {'score': 0.0, 't...",Stephanie Lay said she found the piles of lett...,2,2


now we need only the Ref1/R1, online B from systems and the source. "Online-B.1589" has this structure:
{
    "Online-B.1589": {
        "tgt": "Kmenové buňky 56 pacientů...",
        "score": -0.4510520479
    }
}

so we need to extract the tgt key in our new dataset

In [12]:
# Create new dataframe
new_df = df[["ref", "systems", "src", "document_id", "line_id"]].copy()

# Extract R1 from ref
new_df["ref"] = df["ref"].apply(
    lambda x: x["R1"]
)

# Extract tgt from online-B
new_df["sys"] = df["systems"].apply(
    lambda x: x["Online-B.1589"]["tgt"]
)

# Keep only the columns we want
new_df = new_df[
    ["ref", "sys", "src", "document_id", "line_id"]
]

# Display
print(new_df.head())



                                                 ref  \
0  Vláda přikázala odchod do důchodu dalším 15 da...   
1  Ústřední rada nepřímých daní a cel (CBIC) – ag...   
2  Od června je to už počtvrté, co jsou zkorumpov...   
3  Podle oficiálních zdrojů proběhla akce v návaz...   
4  „Nedávno jsme podnikli odvážný krok, když jsme...   

                                                 sys  \
0  Vláda povinně odešla do důchodu dalších 15 daň...   
1  Ústřední rada nepřímých daní a cel (CBIC) - ag...   
2  Od června je to čtvrté kolo vyhození zkorumpov...   
3  Zdroj uvedl, že akce byla v souladu s adresou ...   
4  "Nedávno jsme podnikli odvážný krok povinného ...   

                                                 src  document_id  line_id  
0  The government has compulsorily retired 15 mor...            1        1  
1  The Central Board of Indirect Taxes and Custom...            1        2  
2  Since June, this is the fourth round of sackin...            1        3  
3  Sources said th

check the created dataset is correct one

In [13]:
new_df.head(50)

,ref,sys,src,document_id,line_id
0,Vláda přikázala odchod do důchodu dalším 15 da...,Vláda povinně odešla do důchodu dalších 15 daň...,The government has compulsorily retired 15 mor...,1,1
1,Ústřední rada nepřímých daní a cel (CBIC) – ag...,Ústřední rada nepřímých daní a cel (CBIC) - ag...,The Central Board of Indirect Taxes and Custom...,1,2
2,"Od června je to už počtvrté, co jsou zkorumpov...",Od června je to čtvrté kolo vyhození zkorumpov...,"Since June, this is the fourth round of sackin...",1,3
3,Podle oficiálních zdrojů proběhla akce v návaz...,"Zdroj uvedl, že akce byla v souladu s adresou ...",Sources said the action was in line with Prime...,1,4
4,"„Nedávno jsme podnikli odvážný krok, když jsme...","""Nedávno jsme podnikli odvážný krok povinného ...","""We have recently taken the bold step of compu...",1,5
5,"Téměř polovina úředníků, kteří odešli do důcho...","Téměř polovina úředníků v důchodu byli ti, kte...",Nearly half of the officials retired were thos...,1,6
6,V červnu vláda donutila odejít do důchodu 15 ú...,V červnu vláda povinně odešla do důchodu 15 úř...,"In June, the government had compulsorily retir...",1,7
7,V srpnu vláda poslala do důchodu 22 úředníků C...,V srpnu vláda povinně odešla do důchodu 22 úře...,"In August, the government compulsorily retired...",1,8
8,Žena v Maine obdržela během pěti dnů 500 dopis...,Žena v Maine dostala 500 dopisů od United Heal...,A woman in Maine got 500 letters from United H...,2,1
9,"Stephanie Layová uvedla, že během čtvrtka a po...","Stephanie Lay uvedla, že ve čtvrtek a pondělí ...",Stephanie Lay said she found the piles of lett...,2,2


**Now check the length of each document**

In [14]:
df_doc_len = (
    new_df.groupby("document_id")["sys"]
      .agg(list)
      .reset_index()
)



In [15]:
df_doc_len.head(20)

,document_id,sys
0,1,[Vláda povinně odešla do důchodu dalších 15 da...
1,2,[Žena v Maine dostala 500 dopisů od United Hea...
2,3,[Využití dat a umělé inteligence k vyzkoušení ...
3,4,[Krátce po zatčení byl proveden příkaz k prohl...
4,5,[Kmenové buňky 56 pacientů s rakovinou u dětí ...
5,6,[Tři skotští studenti byli zařazeni mezi nejle...
6,7,[Investice LNG dosáhly rekordních výsledků v r...
7,8,[Vojenské vidí frustrující trend jako sebevraž...
8,9,"[Sony, Disney zpátky do práce na třetím filmu ..."
9,10,"[All England Club, který pořádá turnaj ve Wimb..."


In [16]:
document_id_list = []
document_len_list = []
for index, row in df_doc_len.iterrows():
    #print(index)
    #print(row["document_id"])
    #print(row["sys"])
    doc_len = 0
    for sys in row["sys"]:
        doc_len += len(sys) 
    if doc_len > 1300 and doc_len <1800:
        #print(row['sys'])
        print(row["document_id"])
        print(doc_len)
        document_id_list.append(row["document_id"])
        document_len_list.append(doc_len)
        

    

3
1785
5
1575
6
1367
7
1625
13
1525
17
1472
18
1523
19
1356


In [17]:
print(document_id_list)
print(document_len_list)

[3, 5, 6, 7, 13, 17, 18, 19]
[1785, 1575, 1367, 1625, 1525, 1472, 1523, 1356]


In [18]:
for index, row in df_doc_len.iterrows():
    #print(index)
    #print(row["document_id"])
    #print(row["sys"])
    doc_len = 0
    for sys in row["sys"]:
        doc_len += len(sys) 
    
    print(doc_len)

2000
1085
1785
884
1575
1367
1625
2088
1020
1271
1019
1169
1525
2443
1252
2477
1472
1523
1356
2096


In [19]:
# Keep only the selected document IDs
df_8_abstracts_with_len = df_doc_len[
    df_doc_len["document_id"].isin(document_id_list)
].copy()

# Add the document length
df_8_abstracts_with_len["document_len"] = df_8_abstracts_with_len["document_id"].map(
    dict(zip(document_id_list, document_len_list))
)

In [20]:
df_8_abstracts_with_len

,document_id,sys,document_len
2,3,[Využití dat a umělé inteligence k vyzkoušení ...,1785
4,5,[Kmenové buňky 56 pacientů s rakovinou u dětí ...,1575
5,6,[Tři skotští studenti byli zařazeni mezi nejle...,1367
6,7,[Investice LNG dosáhly rekordních výsledků v r...,1625
12,13,[Jonathan Van Ness právě vyrazil s Nancy Pelos...,1525
16,17,[„Čína byla vždy odhodlána řešit spory o územn...,1472
17,18,[Indická monzunová sezóna se překročila téměř ...,1523
18,19,"[USA vysílají do Saúdské Arábie vojáky, systém...",1356


In [21]:
for index, row in df_8_abstracts_with_len.iterrows():
    doc_len = 0
    for line in row['sys']:
        doc_len+= len(line)
    print(doc_len)

1785
1575
1367
1625
1525
1472
1523
1356


**next steps**

We calculated oracle error spans for translators postedited version instead of community version. 

- [x] make sure this by loading the translator dataset and finding the **translation_id** there.
    
    Findings: yes they are from translators dataset

1. pass these selected documents through xcomet xl
2. count how many spans there are
3. then calculate the ideal spans
4. if both contains more than 12 spans for each document, prepare everything else.

In [22]:
#check if the translations are from translator dataset or not
import pandas as pd
post_edit_file = "/storage/brno2/home/rahmang/xcomet/arafat_comet/COMET_GR/postedition_aligned.final.translator.tsv"
# Read TSV file into DataFrame
df_translators = pd.read_csv(post_edit_file, sep='\t')
abstracts = [2433, 935, 18234, 18608]
df_translators[df_translators["Translation_id"].isin(abstracts)].drop_duplicates(subset="Translation_id")

,id_hal,Translation_id,line_id,source,translation,postedition
894,1011059,2433,0,Correcting and Validating Syntactic Dependency...,Correction et validation de la dépendance synt...,Correction et validation de la dépendance synt...
923,3986142,18608,0,Exploring Category Structure with Contextual L...,Exploration de la structure des catégories à l...,Exploration de la structure syntaxique à l'aid...
1009,3727214,935,0,Effectiveness of French Language Models on Abs...,Efficacité des modèles de langue française sur...,Efficacité des modèles de langue française sur...
1115,3248881,18234,0,BERT-based Semantic Model for Rescoring N-best...,Modèle sémantique basé sur BERT pour le recala...,Modèle sémantique fondé sur BERT pour la rééva...


### calculate the spans using xcomet

**create data list for calculating the spans without the ref**


In [23]:
# add ref and sources for the selected documents 
grouped = new_df.groupby("document_id").agg({"ref": list, "src": list}).reset_index()
df_8_abstracts_with_len = df_8_abstracts_with_len.merge(grouped, on="document_id", how="left")

In [24]:
df_8_abstracts_with_len.head(8)

,document_id,sys,document_len,ref,src
0,3,[Využití dat a umělé inteligence k vyzkoušení ...,1785,[Využití dat a umělé inteligence ke zvýšení vý...,[Using data and artificial intelligence to try...
1,5,[Kmenové buňky 56 pacientů s rakovinou u dětí ...,1575,[Kmenové buňky 56 dětských pacientů s rakovino...,[Stem cells of 56 child cancer patients lost a...
2,6,[Tři skotští studenti byli zařazeni mezi nejle...,1367,[Tři skotští studenti byli zařazeni mezi nejle...,[Three Scottish students named among Europe's ...
3,7,[Investice LNG dosáhly rekordních výsledků v r...,1625,[Investice do LNG dosáhly v roce 2019 rekordní...,[LNG investments hit record in 2019 & the bigg...
4,13,[Jonathan Van Ness právě vyrazil s Nancy Pelos...,1525,[Jonathan Van Ness se setkal s Nancy Pelosiovo...,[Jonathan Van Ness Just Hung Out With Nancy Pe...
5,17,[„Čína byla vždy odhodlána řešit spory o územn...,1472,[„Čína se vždy snažila řešit spory o územní i ...,"[""China has always been dedicated to resolving..."
6,18,[Indická monzunová sezóna se překročila téměř ...,1523,[Monzunové období v Indii se prodloužilo téměř...,[India's monsoon season has overrun by almost ...
7,19,"[USA vysílají do Saúdské Arábie vojáky, systém...",1356,[USA posílají do Saúdské Arábie vojenské jedno...,"[US sends troops, air defense systems to Saudi..."


In [41]:
data = []
count = 0
for index, row in df_8_abstracts_with_len.iterrows():
    for src_line, sys_line in zip(row["src"], row["sys"]): # all sys, ref, and src have same number of lines
        data.append({
            "src": src_line,
            "mt": sys_line
            
        }
        )


In [26]:
data[0]

{'src': "Using data and artificial intelligence to try and boost revenues is part of HSBC's broader push to squeeze more out of its large physical network and client data, a key priority for interim Chief Executive Noel Quinn.",
 'mt': 'Využití dat a umělé inteligence k vyzkoušení a zvýšení výnosů je součástí širšího tlaku HSBC na vytlačení více z jeho rozsáhlé fyzické sítě a klientských dat, což je klíčová priorita dočasného generálního ředitele Noela Quinna.'}

In [27]:
# load the xcomet custom class and initialize it with xcomet model
#These lines allow the file to load the COMET_GR model - starts here
import sys
from pathlib import Path

COMET_GR = Path.cwd().parent
sys.path.insert(0, str(COMET_GR))
#These lines allow the file to load the COMET_GR model - ends here

# load the custom class without initializing it.
from comet import download_model, load_from_checkpoint
from comet.models.multitask.unified_metric import UnifiedMetric

class CustomXCOMET(UnifiedMetric):
    print("custom unified_metric")

custom unified_metric


In [28]:
#these checks if we loaded the correct COMET codes and where it is located and are we getting 
#the unified_metric class correctly. 
import comet

print("COMET:", comet.__file__)
print("UnifiedMetric:", UnifiedMetric.__module__)

COMET: /auto/brno2/brno2/rahmang/xcomet/arafat_comet/COMET_GR/comet/__init__.py
UnifiedMetric: comet.models.multitask.unified_metric


In [29]:
# here it download the model and load it from the downloaded checkpoint
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/XCOMET-XL")
model = load_from_checkpoint(model_path)
# Here we are initializing the custom class using the loaded checkpoint, so the model is downloaded and
#it will get initialized using the custom class.
model = CustomXCOMET.load_from_checkpoint(model_path,strict = False)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Encoder model frozen.
/storage/brno2/home/rahmang/envs/xcomet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
Encoder model frozen.


In [30]:
data = [
    {
        "src": "Boris Johnson teeters on edge of favour with Tory MPs",
        "mt": "Boris Johnsons Beliebtheit bei Tory-Abgeordneten völlig in der Gunst",
        "ref": "Boris Johnsons Beliebtheit bei Tory-MPs steht auf der Kippe"
    }
]

model_output = model.predict(data, batch_size=8, gpus=1)
# Segment-level scores
print (model_output.scores)

# System-level score
print (model_output.system_score)

# Score explanation (error spans)
print (model_output.metadata.error_spans)

/storage/brno2/home/rahmang/envs/xcomet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 2 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A40') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [GPU-a499a4

[0.6540005207061768]
0.6540005207061768
[[{'text': ['Beliebtheit'], 'confidence': 0.3624686002731323, 'severity': 'major', 'start': 14, 'end': 26, 'check severity': ['major', 'major', 'major', 'major']}, {'text': ['Tory-Abgeordneten', 'völlig', 'in', 'der', 'Gunst'], 'confidence': 0.3602065443992615, 'severity': 'critical', 'start': 30, 'end': 68, 'check severity': ['minor', 'minor', 'minor', 'minor', 'major', 'critical', 'major', 'critical', 'critical']}]]


**now compute the spans for our en_cz dataset**

In [42]:
model_output_en_cz = model.predict(data, batch_size=8, gpus=1)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [GPU-a499a47b-6eca-00c0-ef67-07b64dce9e09]
/storage/brno2/home/rahmang/envs/xcomet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 2 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | fa

In [43]:
print(len(model_output_en_cz.scores)) #we should get 64 outputs as we have 64 lines

64


In [44]:
print (model_output_en_cz.metadata.error_spans[0]) # check the spans for line 1

[{'text': ['k', 'vyzkoušení', 'a', 'zvýšení', 'výnosů'], 'confidence': 0.4998055100440979, 'severity': 'major', 'start': 31, 'end': 61, 'check severity': ['major', 'major', 'major', 'major', 'major', 'major', 'major', 'major']}, {'text': ['tlaku'], 'confidence': 0.4838374853134155, 'severity': 'major', 'start': 81, 'end': 87, 'check severity': ['major']}, {'text': ['na', 'vytlačení', 'více'], 'confidence': 0.4628485143184662, 'severity': 'major', 'start': 92, 'end': 110, 'check severity': ['major', 'major', 'major', 'major', 'major']}]


In [45]:
# now calculate how many spans each line has

print(type(model_output_en_cz.metadata.error_spans)) # should be a list of dict

<class 'list'>


In [46]:
number_of_spans_list = []
for line in model_output_en_cz.metadata.error_spans:
    number_of_spans = 0
    for span in line:
        number_of_spans += len(span['text'])
    number_of_spans_list.append(number_of_spans)

In [47]:
print(number_of_spans_list[0])

9


In [48]:
print(number_of_spans_list)

[9, 12, 6, 5, 12, 8, 6, 15, 11, 0, 0, 7, 7, 9, 14, 11, 0, 2, 8, 14, 2, 0, 7, 8, 2, 1, 13, 6, 2, 3, 1, 4, 4, 27, 1, 1, 26, 17, 4, 10, 7, 3, 23, 5, 1, 28, 4, 3, 3, 5, 4, 6, 3, 1, 15, 5, 0, 0, 1, 8, 0, 12, 1, 3]


In [49]:
len(number_of_spans_list)

64

In [50]:
# now calculate total number of spans in each doc
spans_per_doc = [sum(number_of_spans_list[i:i+8]) for i in range(0, len(number_of_spans_list), 8)]

In [51]:
spans_per_doc

[73, 59, 41, 32, 90, 74, 42, 25]